### Problem Statement 3F

3F. Using the Sign Language MNIST dataset, select any one of the 24 alphabet classes (label Y-24) as an anomaly. Train an Autoencoder Decoder model using the remaining 23 classes and detect the selected class as an anomaly based on reconstruction error.

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
# Load dataset and select class 24 (Letter 'Y') as an anomaly
train_path = "../LP IV Updated Datasets/AutoEncoder/Sign Language MNIST/sign_mnist_train.csv"
test_path = "../LP IV Updated Datasets/AutoEncoder/Sign Language MNIST/sign_mnist_test.csv"

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

# Normalize pixel values
X_train_all = train_data.drop("label", axis=1).values / 255.0
y_train_all = train_data["label"].values

X_test_all = test_data.drop("label", axis=1).values / 255.0
y_test_all = test_data["label"].values

# Select class 24 as the anomaly class
ANOMALY_CLASS = 24

# Train ONLY on the remaining 23 normal classes (excluding class 24)
X_train_normal = X_train_all[y_train_all != ANOMALY_CLASS]

# Test set: normal classes (0) vs class 24 anomaly (1)
X_test = X_test_all
y_test = (y_test_all == ANOMALY_CLASS).astype(int)

print("Normal training images:", X_train_normal.shape[0])
print(f"Test images: {X_test.shape[0]} (Class 24 count: {sum(y_test)})")

In [ ]:
# Build Autoencoder Decoder model
model = Sequential()
# Encoder
model.add(Dense(128, activation='relu', input_shape=(784,)))
model.add(Dense(64, activation='relu'))  # Bottleneck
# Decoder
model.add(Dense(128, activation='relu'))
model.add(Dense(784, activation='sigmoid'))

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
# Train Autoencoder on the 23 normal classes
history = model.fit(X_train_normal, X_train_normal, epochs=5, batch_size=64, validation_split=0.1)

In [ ]:
# Detect class 24 as an anomaly based on reconstruction error
predictions = model.predict(X_test)

# Calculate MSE reconstruction error for each image
mse = np.mean(np.square(X_test - predictions), axis=1)

# Set threshold at 95th percentile of normal test images
threshold = np.percentile(mse[y_test == 0], 95)
print("Anomaly threshold:", threshold)

# Predict anomalies (error > threshold is detected as class 24 anomaly)
y_pred = (mse > threshold).astype(int)

from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal 23 Classes', 'Class 24 (Anomaly)']))

# Plot error distribution
plt.figure(figsize=(8, 4))
plt.hist(mse[y_test == 0], bins=30, alpha=0.6, label='Normal 23 Classes', color='blue')
plt.hist(mse[y_test == 1], bins=30, alpha=0.6, label='Class 24 Anomaly', color='red')
plt.axvline(threshold, color='black', linestyle='--', label='Threshold')
plt.title('Reconstruction Error: Normal Classes vs Class 24')
plt.xlabel('MSE')
plt.ylabel('Count')
plt.legend()
plt.show()